In [ ]:
!rm -rf ~/.cache/huggingface/evaluate

In [1]:
!pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q transformers datasets seqeval evaluate tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00


Evaluation on SE

In [11]:
# =========================================
# 🧪 TEST + ENSEMBLE EVALUATION (CLEAN FIXED VERSION)
# =========================================
import json
import numpy as np
from tqdm import tqdm
import evaluate

seqeval_metric = evaluate.load("seqeval")

test_file_path = "/content/SE-cleaned_file.jsonl"

# =========================================
# 1️⃣ Load Data
# =========================================
dataset = []
with open(test_file_path, "r") as f:
    for line in f:
        dataset.append(json.loads(line))

texts = [normalize(d["input"]) for d in dataset]
true_entities = [json.loads(d["output"]) for d in dataset]

# =========================================
# 2️⃣ Ground Truth → BIO labels
# =========================================
true_token_labels = [encode_labels(t, e) for t, e in zip(texts, true_entities)]

# IMPORTANT FIX:
# already BIO strings → DO NOT use id2label
true_labels_str = true_token_labels

# =========================================
# 3️⃣ ENSEMBLE PREDICTIONS
# =========================================
all_pred_labels = []
all_ents = []

for text in tqdm(texts):
    sci_all, rob_all = [], []

    for chunk in chunk_text(text):
        sci_all.extend(sci_pipe(chunk))
        rob_all.extend(rob_pipe(chunk))

    ents = ensemble(sci_all, rob_all)
    all_ents.append(ents)

    # BIO prediction (already strings)
    pred_labels = encode_labels(text, ents)
    pred_labels_str = pred_labels   # NO id2label

    all_pred_labels.append(pred_labels_str)

# =========================================
# 4️⃣ Align lengths
# =========================================
aligned_preds = []
aligned_truths = []

for pred, true in zip(all_pred_labels, true_labels_str):
    min_len = min(len(pred), len(true))
    aligned_preds.append(pred[:min_len])
    aligned_truths.append(true[:min_len])

# =========================================
# 5️⃣ Evaluation
# =========================================
results = seqeval_metric.compute(
    predictions=aligned_preds,
    references=aligned_truths
)

print("\n📊 ENSEMBLE RESULTS ON NEW FILE:")
print(f"Precision : {results['overall_precision']:.4f}")
print(f"Recall    : {results['overall_recall']:.4f}")
print(f"F1 Score  : {results['overall_f1']:.4f}")
print(f"Accuracy  : {results['overall_accuracy']:.4f}")

# =========================================
# 6️⃣ Save Predictions
# =========================================
output = []

for text, ents in zip(texts, all_ents):
    output.append({
        "text": text,
        "predicted_entities": ents
    })

with open("NEW_FILE_PREDICTIONS.json", "w") as f:
    json.dump(output, f, indent=4)

print("\n✅ Saved: NEW_FILE_PREDICTIONS.json")

100%|██████████| 30/30 [00:00<00:00, 40.40it/s]


📊 ENSEMBLE RESULTS ON NEW FILE:
Precision : 0.6190
Recall    : 0.0844
F1 Score  : 0.1486
Accuracy  : 0.8210

✅ Saved: NEW_FILE_PREDICTIONS.json



/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


SE relatio evaluation

In [21]:
# =========================================
# 🧪 RELATION EVAL + PREDICTION PIPELINE
# =========================================

import json
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# =========================================
# 1️⃣ LOAD RELATION MODEL (already trained)
# =========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# =========================================
# 2️⃣ LABEL MAP
# =========================================
# (already defined in your training code)
# label2id, id2label, tokenizer must exist

# =========================================
# 3️⃣ LOAD DATASETS
# =========================================

# evaluation dataset (GROUND TRUTH)
with open("/content/SE_textDocx_relations.json") as f:
    eval_data = json.load(f)

# entity prediction file (NER output)
with open("/content/NEW_FILE_PREDICTIONS.json") as f:
    ner_data = json.load(f)

# =========================================
# 4️⃣ CLEAN ENTITIES
# =========================================
def clean_entities(entities):
    cleaned = []
    for e in entities:
        text = e["entity"].strip().lower()
        if len(text) < 3:
            continue
        if text in ["the", "this", "that", "data"]:
            continue
        cleaned.append(e)
    return cleaned

# =========================================
# 5️⃣ RELATION PREDICT FUNCTION
# =========================================
def predict_relation(text, e1, t1, e2, t2):
    inp = f"Sentence: {text} Entity1: {e1} ({t1}). Entity2: {e2} ({t2})."

    inputs = tokenizer(
        inp,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        out = model(**inputs)
        probs = torch.softmax(out.logits, dim=1)
        conf, pred = torch.max(probs, dim=1)

    return id2label[pred.item()]

# =========================================
# 6️⃣ EVALUATION LOOP
# =========================================
y_true = []
y_pred = []

for eval_item, ner_item in tqdm(zip(eval_data, ner_data), total=len(eval_data)):

    text = eval_item["input"]
    gold_rel = eval_item["output"].strip()

    entities = clean_entities(ner_item["predicted_entities"])

    predicted_rel = "no_relation"

    # build entity pairs
    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):

            e1 = entities[i]["entity"]
            t1 = entities[i]["label"]
            e2 = entities[j]["entity"]
            t2 = entities[j]["label"]

            rel = predict_relation(text, e1, t1, e2, t2)

            if rel != "no_relation":
                predicted_rel = rel
                break

        if predicted_rel != "no_relation":
            break

    y_true.append(gold_rel)
    y_pred.append(predicted_rel)

# =========================================
# 7️⃣ METRICS
# =========================================
labels = sorted(list(set(y_true + y_pred)))

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average="weighted", zero_division=0
)

acc = accuracy_score(y_true, y_pred)

print("\n📊 RELATION EVALUATION RESULTS")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

# =========================================
# 8️⃣ SAVE FINAL RELATION PREDICTIONS
# =========================================
output = []

for eval_item, ner_item, pred in zip(eval_data, ner_data, y_pred):

    output.append({
        "text": eval_item["input"],
        "gold_relation": eval_item["output"],
        "predicted_relation": pred
    })

with open("FINAL_RELATION_RESULTS.json", "w") as f:
    json.dump(output, f, indent=4)

print("\n✅ Saved: FINAL_RELATION_RESULTS.json")

100%|██████████| 26/26 [00:00<00:00, 567.39it/s]


📊 RELATION EVALUATION RESULTS
Accuracy : 0.2692
Precision: 0.2092
Recall   : 0.2692
F1 Score : 0.1509

✅ Saved: FINAL_RELATION_RESULTS.json
